In [28]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\adaly\OneDrive\Documents\DMDeficientGalaxyTNG-1")
CATALOG = PROJECT_DIR / "data" / "dmdg_catalog_z0.csv"

df = pd.read_csv(CATALOG)
print(f"Total Catalog Objects: {len(df):,}")
resolved = df[
    (df["N_star"] >= 500) &
    (df["SubhaloFlag"] == 1) &
    (df["f_DM"].notna())
].copy()

print(f"Resolved galaxies: {len(resolved):,}")

resolved["M_star_Msun"] = (
    resolved["M_star"] * 1e10 / 0.6774
)

mass_selected = resolved[
    (resolved["M_star_Msun"] > 1e9) &
    (resolved["M_star_Msun"] < 1e10)
].copy()
satellites = mass_selected[
    mass_selected["IsSatellite"] == True
].copy()
satellites["Host_M200c_Msun"] = (
    satellites["Host_M200c"] * 1e10 / 0.6774
)
group_selected = satellites[
    satellites["Host_M200c_Msun"] > 1e13
].copy()
dmdgs = group_selected[
    group_selected["f_DM"] < 0.5
].copy()
dmdgs_non_extreme = dmdgs[
   dmdgs["f_DM"] > 0.05
].copy()
print(f"DMDGS (non-tidal stripping): {len(dmdgs_non_extreme):,}")

Total Catalog Objects: 13,922,457
Resolved galaxies: 154,638
DMDGS (non-tidal stripping): 400


In [29]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, ConfusionMatrixDisplay
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from scipy.stats import randint
from sklearn.tree import export_graphviz
from IPython.display import Image
import graphviz

In [30]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
controls = group_selected[
    group_selected["f_DM"] >= 0.5
].copy()

dmdgs = dmdgs_non_extreme.copy()
controls["logMstar"] = np.log10(controls["M_star"])
controls["logMhost"] = np.log10(controls["Host_M200c_Msun"])

dmdgs["logMstar"] = np.log10(dmdgs["M_star"])
dmdgs["logMhost"] = np.log10(dmdgs["Host_M200c_Msun"])
match_features = ["logMstar", "logMhost"]
scaler = StandardScaler()

X_control = scaler.fit_transform(
    controls[match_features]
)

X_dmdg = scaler.transform(
    dmdgs[match_features]
)
nn = NearestNeighbors(n_neighbors=3)
nn.fit(X_control)

distances, indices = nn.kneighbors(X_dmdg)

print("Matching successful!")
print("DMDGs:", len(dmdgs))
print("Controls:", len(controls))
print("Distance array shape:", distances.shape)
print("Index array shape:", indices.shape)

Matching successful!
DMDGs: 400
Controls: 15082
Distance array shape: (400, 3)
Index array shape: (400, 3)


In [31]:
print(dmdgs)
print(controls)
fullDataSet = pd.concat([dmdgs, controls], ignore_index=True)
print(fullDataSet)

         SubhaloID  GroupID  M_DM_2Rh  M_total_2Rh      f_DM    M_star  \
659            659        0  0.159337     0.585692  0.272049  0.561757   
12084        12084        1  0.063735     0.424455  0.150157  0.565132   
18203        18203        2  0.382409     0.795851  0.480504  0.483622   
18287        18287        2  0.043818     0.368323  0.118966  0.373522   
18310        18310        2  0.187221     0.410042  0.456590  0.300666   
...            ...      ...       ...          ...       ...       ...   
1122765    1122765     3981  0.314691     0.630473  0.499134  0.315783   
1138270    1138270     4174  0.191205     0.463710  0.412337  0.361777   
1139036    1139036     4184  0.513862     1.210406  0.424537  0.652038   
1154202    1154202     4390  0.211122     0.492998  0.428240  0.302069   
1168747    1168747     4593  0.227055     0.534112  0.425108  0.284097   

            M_gas  N_gas  N_DM  N_star  ...  Host_R200c  IsCentral  \
659      0.000000      0    65    1125  .

In [32]:
from sklearn.model_selection import train_test_split
fullDataSet["gas_fraction"] = (
    fullDataSet["M_gas"] /
    (fullDataSet["M_star"] + fullDataSet["M_gas"])
)
features = [
    "logMstar",
    "logMhost",
    "r_over_R200c",
    "gas_fraction",
    "SFR",
    "Vmax"
]

X = fullDataSet[features]
Y = fullDataSet["f_DM"]
X_train, X_test, y_train, y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42
)


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("Random Forest results:")
print(f"R²   = {r2:.4f}")
print(f"MAE  = {mae:.4f}")
print(f"RMSE = {rmse:.4f}")


Random Forest results:
R²   = 0.4844
MAE  = 0.0497
RMSE = 0.0734


In [59]:
predictions = X_test.copy()
predictions["actual_f_DM"] = y_test
predictions["predicted_f_DM"] = y_pred
predictions["residual"] = (
    predictions["actual_f_DM"] -
    predictions["predicted_f_DM"]
)
predictionsDMDG = predictions[
   predictions["actual_f_DM"] < 0.5
].copy()
print(predictionsDMDG["residual"])
averageNonAbsolute = predictionsDMDG["residual"].median()
print(averageNonAbsolute)
predictionsNonDMDG = predictions[
   predictions["actual_f_DM"] > 0.5
]
print(predictionsNonDMDG["residual"]) 
averageNonAbsolute = predictionsNonDMDG["residual"].median()
print(averageNonAbsolute)

150   -0.350643
251   -0.342041
99    -0.169440
123   -0.220750
19    -0.315237
         ...   
219   -0.188155
59    -0.078396
398   -0.001325
330    0.013807
335   -0.178161
Name: residual, Length: 92, dtype: float64
-0.17658211895999926
10694    0.027480
15149   -0.017649
13445   -0.041187
9381     0.079167
9188    -0.007988
           ...   
12093    0.056375
6126     0.058984
7586     0.005357
8298    -0.122882
15422   -0.004650
Name: residual, Length: 3005, dtype: float64
0.009157491319997324


In [62]:
import pandas as pd

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "Importance",
    ascending=False
)

print(feature_importance)

        Feature  Importance
2  r_over_R200c    0.425234
5          Vmax    0.179539
0      logMstar    0.178065
1      logMhost    0.110376
4           SFR    0.074407
3  gas_fraction    0.032380
